# Task 4 Resume — new GPU account (clone + Drive)

Use this on a **different Google account** that still has Colab GPU.

**What happened:** account-A GPU died near Vanilla epoch 98.  
**What we keep:** `vanilla_best.pt` (val Acc ≈ 0.9544) — **do not retrain Vanilla**.

This notebook will:
1. Mount **this** account’s Drive (for durable results)
2. Clone / pull `ATML-PA1` from GitHub into `/content`
3. Point `task4/results` → Drive (so disconnects don’t wipe outputs)
4. Restore the Vanilla resume bundle (upload / gdown / Drive path)
5. Continue **live**: GCSC → PROSER → extract → eval

**Before you start:** upload `task4_resume_from_vanilla.zip` to this account’s Drive  
(from account A: `ATML-PA1/task4/task4_resume_from_vanilla.zip`), **or** set a gdown file id below.

In [ ]:
import torch
print("cuda:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit("Runtime → Change runtime type → T4 GPU, then re-run.")
print("device:", torch.cuda.get_device_name(0))

## 1) Mount this account’s Drive

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/ATML-PA1-task4-backup")
DRIVE_RESULTS = DRIVE_ROOT / "results"
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)
print("Drive results:", DRIVE_RESULTS)

## 2) Clone / pull GitHub repo into `/content`

In [ ]:
from pathlib import Path
import os, subprocess

REPO_URL = "https://github.com/ttqureshi/ATML-PA1.git"
REPO_DIR = Path("/content/ATML-PA1")

if (REPO_DIR / ".git").is_dir():
    subprocess.check_call(["git", "-C", str(REPO_DIR), "fetch", "origin"])
    subprocess.check_call(["git", "-C", str(REPO_DIR), "checkout", "main"])
    subprocess.check_call(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", "main"])
else:
    subprocess.check_call(["git", "clone", "--branch", "main", REPO_URL, str(REPO_DIR)])

os.chdir(REPO_DIR)
print("cwd:", Path.cwd())
print(subprocess.check_output(["git", "log", "-1", "--oneline"], text=True).strip())

In [ ]:
%pip install -q -r requirements.txt

## 3) Symlink `task4/results` → Drive (durable)

In [ ]:
from pathlib import Path
import shutil

REPO_DIR = Path("/content/ATML-PA1")
LOCAL_RESULTS = REPO_DIR / "task4" / "results"
DRIVE_ROOT = Path("/content/drive/MyDrive/ATML-PA1-task4-backup")
DRIVE_RESULTS = DRIVE_ROOT / "results"
DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)

if LOCAL_RESULTS.is_symlink():
    LOCAL_RESULTS.unlink()
elif LOCAL_RESULTS.exists():
    shutil.copytree(LOCAL_RESULTS, DRIVE_RESULTS, dirs_exist_ok=True)
    shutil.rmtree(LOCAL_RESULTS)

LOCAL_RESULTS.parent.mkdir(parents=True, exist_ok=True)
LOCAL_RESULTS.symlink_to(DRIVE_RESULTS)
assert LOCAL_RESULTS.resolve() == DRIVE_RESULTS.resolve()
print("OK:", LOCAL_RESULTS, "→", LOCAL_RESULTS.resolve())

## 4) Restore Vanilla resume bundle

Pick **one** method (A recommended).

In [ ]:
from pathlib import Path
import shutil, zipfile

DRIVE_ROOT = Path("/content/drive/MyDrive/ATML-PA1-task4-backup")
RESULTS = Path("/content/ATML-PA1/task4/results")
RESULTS.mkdir(parents=True, exist_ok=True)

# === Method A: zip already on THIS Drive (upload task4_resume_from_vanilla.zip) ===
CANDIDATES = [
    DRIVE_ROOT / "task4_resume_from_vanilla.zip",
    Path("/content/drive/MyDrive/task4_resume_from_vanilla.zip"),
    Path("/content/task4_resume_from_vanilla.zip"),
]

# === Method B: gdown (optional). Paste file id if you shared the zip "anyone with link" ===
GDOWN_FILE_ID = ""  # e.g. "1abc..." from Drive share link

zip_path = next((p for p in CANDIDATES if p.exists()), None)

if zip_path is None and GDOWN_FILE_ID.strip():
    %pip install -q gdown
    import gdown
    zip_path = Path("/content/task4_resume_from_vanilla.zip")
    gdown.download(id=GDOWN_FILE_ID.strip(), output=str(zip_path), quiet=False)

if zip_path is None:
    raise SystemExit(
        "Resume zip not found. Upload task4_resume_from_vanilla.zip to this Drive "
        "(e.g. MyDrive/ATML-PA1-task4-backup/) or set GDOWN_FILE_ID."
    )

print("Using zip:", zip_path, zip_path.stat().st_size)
with zipfile.ZipFile(zip_path, "r") as zf:
    zf.extractall(RESULTS)

need = [
    RESULTS / "checkpoints" / "vanilla_best.pt",
    RESULTS / "splits" / "cifar10_seed6304.json",
    RESULTS / "tables" / "vanilla_train_summary.json",
]
for p in need:
    print(" ", p.relative_to(RESULTS), "OK" if p.exists() else "MISSING", p.stat().st_size if p.exists() else "")
if not all(p.exists() for p in need):
    raise SystemExit("Resume bundle incomplete")

import torch
meta = torch.load(need[0], map_location="cpu", weights_only=False).get("meta", {})
print("Vanilla meta:", meta)

## 5) Helpers

In [ ]:
from pathlib import Path
import shutil

REPO_DIR = Path("/content/ATML-PA1")
RESULTS = REPO_DIR / "task4" / "results"
DRIVE_ROOT = Path("/content/drive/MyDrive/ATML-PA1-task4-backup")

def stage_done(name: str) -> bool:
    """Skip only if checkpoint AND train summary exist (completed stage)."""
    pt = RESULTS / "checkpoints" / f"{name}_best.pt"
    summary = RESULTS / "tables" / f"{name}_train_summary.json"
    return pt.exists() and summary.exists()

def refresh_bundle(tag: str = ""):
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
    for zip_base in (DRIVE_ROOT / "task4_results_bundle", REPO_DIR / "task4_results_bundle"):
        zp = zip_base.with_suffix(".zip")
        if zp.exists():
            zp.unlink()
        shutil.make_archive(str(zip_base), "zip", root_dir=RESULTS)
        print(f"[{tag}] {zp} ({zp.stat().st_size} bytes)")
    for sub in ["checkpoints", "tables", "curves", "figures", "splits", "cache"]:
        p = RESULTS / sub
        n = sum(1 for _ in p.rglob("*") if _.is_file()) if p.exists() else 0
        print(f"  {sub}: {n} files")

print("vanilla done?", stage_done("vanilla"))
print("gcsc done?", stage_done("gcsc"))
print("proser done?", stage_done("proser"))
refresh_bundle("resume-init")

## 6) LIVE — skip Vanilla, train GCSC

Watch this cell’s output. Keep the tab open.

In [ ]:
import os
from pathlib import Path
os.chdir("/content/ATML-PA1")

if stage_done("gcsc"):
    print("SKIP train_gcsc — already complete on Drive")
else:
    assert stage_done("vanilla"), "Vanilla resume files missing"
    !python -m task4.scripts.run_task4 --stages train_gcsc

refresh_bundle("gcsc")

## 7) LIVE — train PROSER (needs Vanilla ckpt)

In [ ]:
import os
os.chdir("/content/ATML-PA1")

if stage_done("proser"):
    print("SKIP train_proser — already complete on Drive")
else:
    assert (Path("task4/results/checkpoints/vanilla_best.pt")).exists()
    !python -m task4.scripts.run_task4 --stages train_proser

refresh_bundle("proser")

## 8) LIVE — extract logits/features + OSR eval

In [ ]:
import os
os.chdir("/content/ATML-PA1")
!python -m task4.scripts.run_task4 --stages extract_all
refresh_bundle("extract")

In [ ]:
import os
from pathlib import Path
os.chdir("/content/ATML-PA1")
!python -m task4.scripts.run_task4 --stages eval
refresh_bundle("eval")
Path("/content/drive/MyDrive/ATML-PA1-task4-backup/task4_pipeline.done").write_text("DONE\n")
print("ALL DONE — download MyDrive/ATML-PA1-task4-backup/task4_results_bundle.zip back to account A")

## Done — bring results home

On **this** GPU account Drive:
`MyDrive/ATML-PA1-task4-backup/task4_results_bundle.zip`

1. Download / share to account A
2. Unzip into account-A `ATML-PA1/task4/results/` (merge)
3. Tell the agent to verify tables/figures and interpret RQs

If runtime dies mid-GCSC/PROSER: reopen this notebook, remount Drive, re-run from the failed stage — completed stages are skipped via `*_train_summary.json`.